# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

In [ ]:
# import torch
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [ ]:
#import os
#os.environ["PATH"] = f"/home/p2chung/.local/bin:{os.environ['PATH']}"

#!uv venv .venv --seed
#!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter
#!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"
#print("Done.")

### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import json
import os

MODEL_ID      = "Qwen/Qwen3-4B-Thinking-2507"
BASE_MODEL_ID = MODEL_ID
LORA_PATH     = "lora_v2_checkpoints/final"
GPU_ID        = "0"
DATA_PATH     = "data/public.jsonl"
MAX_TOKENS    = 8192
N_SAMPLES     = 1

OUTPUT_PATH = "results/run6_lora_v2_sc3_public_1126.jsonl"

GENERATION_CONFIG = {
    "max_new_tokens": MAX_TOKENS,
    "temperature": 0.6,
    "top_p": 0.95,
    "top_k": 20,
    "repetition_penalty": 1.0,
}

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_DISABLE_DEEP_GEMM"] = "1"


import re
import sys
from pathlib import Path
from typing import Optional
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

In [ ]:
# cutoff_rate = sum("\\boxed" not in r["response"] for r in saved_data) / len(saved_data)
# print(f"Cutoff rate (no boxed answer): {cutoff_rate:.2%}")

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
PROMPT_VERSION = "concise"  # change to "concise" for runs 2, 3, 4

SYSTEM_PROMPT_MATH_STARTER = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)
SYSTEM_PROMPT_MCQ_STARTER = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)
SYSTEM_PROMPT_MATH_CONCISE = (
    "You are an expert mathematician. "
    "Solve the problem step-by-step, but be concise — no repetition, no over-explanation. "
    "Think about the problem type first, then apply the correct method. "
    "Before finalizing, verify your answer is correct. "
    "Your response MUST end with \\boxed{answer} as the absolute last thing you write. "
    "You MUST always select one of the given options — never say none are correct. "
    "For multiple sub-answers in order, use \\boxed{a, b, c}. "
    "Never write anything after the boxed answer."
)
SYSTEM_PROMPT_MCQ_CONCISE = (
    "You are an expert mathematician. "
    "Read the problem and all options carefully. Eliminate wrong answers, then select the best one. "
    "Your response MUST end with \\boxed{X} where X is the letter only. "
    "Never write anything after the boxed answer."
)

SYSTEM_PROMPT_MATH = SYSTEM_PROMPT_MATH_STARTER if PROMPT_VERSION == "starter" else SYSTEM_PROMPT_MATH_CONCISE
SYSTEM_PROMPT_MCQ  = SYSTEM_PROMPT_MCQ_STARTER  if PROMPT_VERSION == "starter" else SYSTEM_PROMPT_MCQ_CONCISE

from typing import Optional

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes",
#     enable_prefix_caching=False,
#     gpu_memory_utilization=0.50,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("Model loaded.")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=BASE_MODEL_ID,
    enable_lora=True,
    max_lora_rank=32,
    dtype="float16",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    trust_remote_code=True,
    disable_log_stats=True,
)

tokenizer = llm.get_tokenizer()
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded.")

In [ ]:
# !.venv/bin/python -m pip install accelerate


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

### Generate with Transformers (for Datahub)

In [ ]:
from vllm.lora.request import LoRARequest
from pathlib import Path
from collections import Counter
import json

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

items = data  # all 1126

completed_ids = set()
if out_path.exists():
    with open(out_path) as f:
        for line in f:
            try:
                completed_ids.add(json.loads(line)["id"])
            except:
                pass
print(f"Already completed: {len(completed_ids)} questions")

pending_items = [item for item in items if item.get("id") not in completed_ids]

if not pending_items:
    print("All done!")
else:
    prompts = []
    for item in pending_items:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for _ in range(N_SAMPLES):
            prompts.append(prompt_text)

    sampling_params = SamplingParams(
        max_tokens=GENERATION_CONFIG["max_new_tokens"],
        temperature=GENERATION_CONFIG["temperature"],
        top_p=GENERATION_CONFIG["top_p"],
        top_k=GENERATION_CONFIG["top_k"],
        repetition_penalty=GENERATION_CONFIG["repetition_penalty"],
    )

    lora_request = LoRARequest("math_lora_v2", 1, LORA_PATH)

    print(f"Generating {N_SAMPLES}x for {len(pending_items)} questions...")
    outputs = llm.generate(prompts, sampling_params, lora_request=lora_request)

    def extract_final_answer(response, is_mcq):
        if is_mcq:
            m = re.search(r"\\boxed\{([A-Za-z])\}", response)
            if m:
                return m.group(1).upper()
            matches = re.findall(r"\b([A-Z])\b", response.upper())
            return matches[-1] if matches else ""
        else:
            m = re.search(r"\\boxed\{([^}]+)\}", response)
            return m.group(1).strip() if m else ""

    with open(out_path, "a") as f:
        for i, item in enumerate(pending_items):
            item_outputs = outputs[i*N_SAMPLES:(i+1)*N_SAMPLES]
            all_responses = [o.outputs[0].text.strip() for o in item_outputs]
            is_mcq = bool(item.get("options"))
            answers = [extract_final_answer(r, is_mcq) for r in all_responses]
            majority_answer = Counter(answers).most_common(1)[0][0] if answers else ""
            best_response = next(
                (r for r in all_responses if extract_final_answer(r, is_mcq) == majority_answer),
                all_responses[0]
            )
            record = {
                "id": item.get("id"),
                "question": item["question"],
                "response": best_response,
                "answer": item.get("answer"),
                "options": item.get("options"),
                "all_answers": answers,
                "majority_answer": majority_answer,
            }
            f.write(json.dumps(record) + "\n")

    boxed_rate = sum(
        "\\boxed" in outputs[i*N_SAMPLES].outputs[0].text
        for i in range(len(pending_items))
    ) / len(pending_items)
    print(f"Boxed rate: {boxed_rate:.2%}")
    print(f"Saved to {out_path}")

In [ ]:
print(outputs[0].outputs[0].text[:500])


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
import json
import re
import sys
from pathlib import Path
from multiprocessing import Process, Queue, set_start_method

try:
    set_start_method("fork")
except RuntimeError:
    pass

sys.path.insert(0, ".")
from judger import Judger

INPUT_PATH  = "results/run6_lora_v2_sc3_public_1126.jsonl"
SCORED_PATH = "results/run6_lora_v2_sc3_scored.jsonl"

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

def judge_worker(q, response, gold_list):
    try:
        from judger import Judger
        j = Judger(strict_extract=False)
        result = j.auto_judge(
            pred=response,
            gold=gold_list,
            options=[[]] * len(gold_list),
        )
        q.put(result)
    except Exception:
        q.put(False)

def safe_judge(response, gold_list, timeout=30):
    q = Queue()
    p = Process(target=judge_worker, args=(q, response, gold_list))
    p.start()
    p.join(timeout=timeout)
    if p.is_alive():
        p.kill()
        p.join()
        return False
    if not q.empty():
        return q.get()
    return False

completed_ids = set()
if Path(SCORED_PATH).exists():
    with open(SCORED_PATH) as f:
        for line in f:
            try:
                completed_ids.add(json.loads(line)["id"])
            except:
                pass
print(f"Already scored: {len(completed_ids)}")

saved_data = []
with open(INPUT_PATH) as f:
    for line in f:
        saved_data.append(json.loads(line))

pending = [r for r in saved_data if r.get("id") not in completed_ids]
print(f"Scoring {len(pending)} remaining...")

with open(SCORED_PATH, "a") as f:
    for i, record in enumerate(pending):
        is_mcq = bool(record.get("options"))
        gold = record["answer"]
        response = record["response"]

        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            correct = safe_judge(response, gold_list, timeout=30)

        result = {
            "id": record.get("id"),
            "is_mcq": is_mcq,
            "gold": gold,
            "response": response,
            "correct": correct,
        }
        f.write(json.dumps(result) + "\n")

        if (i + 1) % 50 == 0:
            print(f"Scored {i+1}/{len(pending)}...")

all_results = []
with open(SCORED_PATH) as f:
    for line in f:
        all_results.append(json.loads(line))

mcq      = [r for r in all_results if r["is_mcq"]]
freeform = [r for r in all_results if not r["is_mcq"]]
print(f"\n{'='*50}")
print(f"EVALUATION RESULTS")
print(f"{'='*50}")
print(f"  MCQ        : {sum(r['correct'] for r in mcq):3d} / {len(mcq):3d}  ({100*sum(r['correct'] for r in mcq)/len(mcq):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in freeform):3d} / {len(freeform):3d}  ({100*sum(r['correct'] for r in freeform)/len(freeform):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in all_results):3d} / {len(all_results):3d}  ({100*sum(r['correct'] for r in all_results)/len(all_results):.2f}%)")
print(f"{'='*50}")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

In [ ]:
boxed_rate = sum("\\boxed" in r["response"] for r in saved_data) / len(saved_data)
print(f"Boxed rate: {boxed_rate:.2%}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

## LoRA fine-tuning


In [ ]:
# ============ PHASE 1 / CELL A: config + helpers (vLLM session) ============
import os, re, json, sys, random
from pathlib import Path
from collections import defaultdict, Counter
from multiprocessing import Process, Queue, set_start_method
try:
    set_start_method("fork")
except RuntimeError:
    pass

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_DISABLE_DEEP_GEMM"] = "1"

MODEL_ID  = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = "data/public.jsonl"
SEED      = 42

RS_K          = 4          # candidates generated per question
RS_MAX_KEEP   = 2          # correct traces kept per question
RS_TEMP       = 0.8
RS_MAX_TOKENS = 8192
RS_OUT        = "data/rs_traces.jsonl"

SPLIT_PATH    = "data/split_ids.json"
VAL_FRAC      = 0.10
TRAIN_JSONL   = "data/lora_train.jsonl"

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Solve the problem step-by-step, but be concise, no repetition, no over-explanation. "
    "Think about the problem type first, then apply the correct method. "
    "Give the final answer in exact form: keep symbolic constants such as pi, e, and square "
    "roots, and exact fractions. Do not convert to decimals. "
    "If the answer is a family of solutions, include the free integer parameter exactly as the "
    "problem names it (for example k). Use the exact notation and variable names the problem uses. "
    "If the problem shows [ANS], your boxed answer is whatever replaces [ANS]. "
    "Before finalizing, verify your answer is correct. "
    "Your response MUST end with \\boxed{answer} as the absolute last thing you write. "
    "For multiple sub-answers in order, use \\boxed{a, b, c}. "
    "Never write anything after the boxed answer."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and all options carefully. Eliminate wrong answers, then select the best one. "
    "You MUST always select one of the given options, never say none are correct. "
    "Your response MUST end with \\boxed{X} where X is the letter only. "
    "Never write anything after the boxed answer."
)

def build_prompt(question, options):
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts = "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts}"
    return SYSTEM_PROMPT_MATH, question

def extract_boxed(text):
    starts = [m.end() for m in re.finditer(r"\\boxed\s*\{", text)]
    if not starts:
        return ""
    i, depth, out = starts[-1], 1, []
    while i < len(text) and depth:
        c = text[i]
        if c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0: break
        out.append(c); i += 1
    return "".join(out).strip()

def extract_mcq_letter(text):
    b = extract_boxed(text)
    m = re.match(r"\s*([A-Za-z])\s*$", b)
    return m.group(1).upper() if m else ""

def _judge_worker(q, response, gold_list):
    try:
        from judger import Judger
        j = Judger(strict_extract=False)
        q.put(j.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list)))
    except Exception:
        q.put(False)

def safe_judge(response, gold_list, timeout=30):
    q = Queue()
    p = Process(target=_judge_worker, args=(q, response, gold_list))
    p.start(); p.join(timeout=timeout)
    if p.is_alive():
        p.kill(); p.join(); return False
    return q.get() if not q.empty() else False

def is_correct(response, item):
    if not extract_boxed(response):
        return False
    gold = item.get("answer")
    if item.get("options"):
        return extract_mcq_letter(response) == str(gold).strip().upper()
    gold_list = gold if isinstance(gold, list) else [gold]
    return bool(safe_judge(response, gold_list))

print("Phase 1 config + helpers ready.")

In [ ]:
# ============ PHASE 1 / CELL B: load data + base model (vLLM) ============
import torch
from vllm import LLM, SamplingParams
print("GPU:", torch.cuda.get_device_name(0))

data  = [json.loads(l) for l in open(DATA_PATH)]
by_id = {d["id"]: d for d in data}
n_mcq = sum(1 for d in data if d.get("options"))
print(f"Loaded {len(data)} questions ({n_mcq} MCQ, {len(data)-n_mcq} free-form)")

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    gpu_memory_utilization=0.95,
    max_model_len=8192,
    enable_prefix_caching=True,
    trust_remote_code=True,
    disable_log_stats=True,
)
tokenizer = llm.get_tokenizer()

def prompt_str_for(item):
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)

print("Base model loaded.")

In [ ]:
test = by_id[0]
print("gold:", test["answer"])
print("judge 105950 ->", is_correct(r"The sum is \boxed{105950}", test))
print("judge 325*326 ->", is_correct(r"\boxed{325 \cdot 326}", test))

In [ ]:
# ============ PHASE 1 / CELL C: train / val split by question id ============
if Path(SPLIT_PATH).exists():
    split = json.load(open(SPLIT_PATH))
    print("Loaded existing split.")
else:
    ids = [d["id"] for d in data]
    random.Random(SEED).shuffle(ids)
    n_val = max(1, int(len(ids) * VAL_FRAC))
    split = {"val": ids[:n_val], "train": ids[n_val:]}
    Path(SPLIT_PATH).parent.mkdir(parents=True, exist_ok=True)
    json.dump(split, open(SPLIT_PATH, "w"))
    print("Created new split.")

train_ids = set(split["train"])
val_ids   = set(split["val"])
print(f"train={len(train_ids)} questions, val={len(val_ids)} questions")

In [ ]:
# ============ PHASE 1 / CELL D: rejection sampling on train ids (resumable) ============
kept = defaultdict(list)
if Path(RS_OUT).exists():
    for l in open(RS_OUT):
        try:
            r = json.loads(l); kept[r["id"]].append(r["completion"])
        except Exception:
            pass
print(f"Resuming: {sum(1 for v in kept.values() if v)} train questions already solved")

todo = [by_id[i] for i in train_ids if len(kept[i]) < RS_MAX_KEEP]
print(f"Sampling {len(todo)} questions, K={RS_K} each")

sp = SamplingParams(n=RS_K, temperature=RS_TEMP, top_p=0.95, top_k=20,
                    max_tokens=RS_MAX_TOKENS, repetition_penalty=1.0)
CHUNK = 64
Path(RS_OUT).parent.mkdir(parents=True, exist_ok=True)
with open(RS_OUT, "a") as fout:
    for s in range(0, len(todo), CHUNK):
        chunk = todo[s:s+CHUNK]
        outs = llm.generate([prompt_str_for(it) for it in chunk], sp)
        for it, o in zip(chunk, outs):
            cands = []
            for c in o.outputs:
                t = c.text.strip()
                if extract_boxed(t) and is_correct(t, it):
                    cands.append(t)
            for t in sorted(set(cands), key=len)[:RS_MAX_KEEP]:   # shortest distinct
                fout.write(json.dumps({"id": it["id"],
                    "prompt": prompt_str_for(it), "completion": t}) + "\n")
        fout.flush()
        print(f"  done {min(s+CHUNK, len(todo))}/{len(todo)}")
print("Rejection sampling complete. Re-run this cell to retry unsolved hard questions.")

In [ ]:
# ============ PHASE 1 / CELL E: build SFT training file ============
recs = [json.loads(l) for l in open(RS_OUT)]
EOS  = tokenizer.eos_token

solved_ids = {r["id"] for r in recs}
mcq_solved = sum(1 for i in solved_ids if by_id[i].get("options"))
print(f"Train questions with a correct trace: {len(solved_ids)} / {len(train_ids)}")
print(f"  MCQ {mcq_solved}, free-form {len(solved_ids)-mcq_solved}")
print(f"Total training examples: {len(recs)}")

with open(TRAIN_JSONL, "w") as f:
    for r in recs:
        f.write(json.dumps({"text": r["prompt"] + r["completion"] + EOS}) + "\n")
print(f"Wrote {TRAIN_JSONL}")

PHASE 2

In [2]:
# ============ PHASE 2 / CELL F: base model + LoRA (training session) ============
import os, torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
OUTPUT_DIR  = "lora_ckpts"
TRAIN_JSONL = "data/lora_train.jsonl"

LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    "lora_dropout": 0.05,
    "bias": "none",
}
TRAIN_CONFIG = {
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 1e-4,
    "max_length": 4096,                # was 8192
    "lr_scheduler_type": "cosine",
    "logging_steps": 10,
    "save_strategy": "epoch",
    "bf16": True,
    "seed": 42,
}

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, attn_implementation="eager")    # was "eager"
model.config.use_cache = False
model.enable_input_require_grads()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = get_peft_model(model, LoraConfig(task_type=TaskType.CAUSAL_LM, **LORA_CONFIG))
model.print_trainable_parameters()
print("LoRA applied.")

In [3]:
from datasets import load_dataset
ds = load_dataset("json", data_files=TRAIN_JSONL, split="train")

def fits(ex):
    return len(tokenizer(ex["text"], add_special_tokens=False)["input_ids"]) <= 4096

ds = ds.filter(fits).shuffle(seed=42)
print(f"{len(ds)} training examples after length filter")
print(ds[0]["text"][-400:])

In [4]:
# ============ PHASE 2 / CELL H: train (one checkpoint saved per epoch) ============
from trl import SFTTrainer, SFTConfig
from pathlib import Path

cps  = sorted(Path(OUTPUT_DIR).glob("checkpoint-*")) if Path(OUTPUT_DIR).exists() else []
ckpt = str(cps[-1]) if cps else None
print(f"Resuming from {ckpt}" if ckpt else "Fresh start")

sft = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=TRAIN_CONFIG["num_train_epochs"],
    per_device_train_batch_size=TRAIN_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAIN_CONFIG["gradient_accumulation_steps"],
    learning_rate=TRAIN_CONFIG["learning_rate"],
    max_length=TRAIN_CONFIG["max_length"],
    lr_scheduler_type=TRAIN_CONFIG["lr_scheduler_type"],
    logging_steps=TRAIN_CONFIG["logging_steps"],
    save_strategy=TRAIN_CONFIG["save_strategy"],
    bf16=TRAIN_CONFIG["bf16"],
    seed=TRAIN_CONFIG["seed"],
    dataset_text_field="text",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_total_limit=4,
    report_to="none",
)
trainer = SFTTrainer(model=model, train_dataset=ds, args=sft, processing_class=tokenizer)
trainer.train(resume_from_checkpoint=ckpt)
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"Saved final adapter to {OUTPUT_DIR}/final")

In [2]:
# ============ PHASE 3 / CELL I: setup + base model with LoRA (vLLM) ============
import os, re, json
from pathlib import Path
from collections import Counter
from multiprocessing import Process, Queue, set_start_method
try:
    set_start_method("fork")
except RuntimeError:
    pass

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_DISABLE_DEEP_GEMM"] = "1"

import torch
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

MODEL_ID       = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH      = "data/public.jsonl"
PRIV_PATH      = "data/private.jsonl"
SPLIT_PATH     = "data/split_ids.json"
OUTPUT_DIR     = "lora_ckpts"
MAX_LORA_RANK  = 16
N_VOTE         = 3              # 3 keeps inference time reasonable
GEN_MAX_TOKENS = 8192

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Solve the problem step-by-step, but be concise, no repetition, no over-explanation. "
    "Think about the problem type first, then apply the correct method. "
    "Give the final answer in exact form: keep symbolic constants such as pi, e, and square "
    "roots, and exact fractions. Do not convert to decimals. "
    "If the answer is a family of solutions, include the free integer parameter exactly as the "
    "problem names it (for example k). Use the exact notation and variable names the problem uses. "
    "If the problem shows [ANS], your boxed answer is whatever replaces [ANS]. "
    "Before finalizing, verify your answer is correct. "
    "Your response MUST end with \\boxed{answer} as the absolute last thing you write. "
    "For multiple sub-answers in order, use \\boxed{a, b, c}. "
    "Never write anything after the boxed answer."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and all options carefully. Eliminate wrong answers, then select the best one. "
    "You MUST always select one of the given options, never say none are correct. "
    "Your response MUST end with \\boxed{X} where X is the letter only. "
    "Never write anything after the boxed answer."
)

def build_prompt(question, options):
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts = "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts}"
    return SYSTEM_PROMPT_MATH, question

def extract_boxed(text):
    starts = [m.end() for m in re.finditer(r"\\boxed\s*\{", text)]
    if not starts:
        return ""
    i, depth, out = starts[-1], 1, []
    while i < len(text) and depth:
        c = text[i]
        if c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0: break
        out.append(c); i += 1
    return "".join(out).strip()

def extract_mcq_letter(text):
    b = extract_boxed(text)
    m = re.match(r"\s*([A-Za-z])\s*$", b)
    return m.group(1).upper() if m else ""

def _judge_worker(q, response, gold_list):
    try:
        from judger import Judger
        j = Judger(strict_extract=False)
        q.put(j.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list)))
    except Exception:
        q.put(False)

def safe_judge(response, gold_list, timeout=30):
    q = Queue()
    p = Process(target=_judge_worker, args=(q, response, gold_list))
    p.start(); p.join(timeout=timeout)
    if p.is_alive():
        p.kill(); p.join(); return False
    return q.get() if not q.empty() else False

def is_correct(response, item):
    if not extract_boxed(response): return False
    gold = item.get("answer")
    if item.get("options"):
        return extract_mcq_letter(response) == str(gold).strip().upper()
    gl = gold if isinstance(gold, list) else [gold]
    return bool(safe_judge(response, gl))

data  = [json.loads(l) for l in open(DATA_PATH)]
by_id = {d["id"]: d for d in data}
split = json.load(open(SPLIT_PATH))
val_items = [by_id[i] for i in split["val"]]
print(f"val = {len(val_items)} questions")

llm = LLM(model=MODEL_ID, enable_lora=True, max_lora_rank=MAX_LORA_RANK,
          dtype="bfloat16", gpu_memory_utilization=0.90, max_model_len=8192,
          enable_prefix_caching=True, trust_remote_code=True, disable_log_stats=True)
tokenizer = llm.get_tokenizer()

def prompt_str_for(item):
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)

def vote(samples, item):
    keyf = extract_mcq_letter if item.get("options") else extract_boxed
    answers = [keyf(s) for s in samples]
    valid = [a for a in answers if a]
    if not valid:
        return samples[0]
    maj = Counter(valid).most_common(1)[0][0]
    for s, a in zip(samples, answers):
        if a == maj:
            return s
    return samples[0]

print("Phase 3 ready.")

val = 112 questions
INFO 06-01 01:02:03 [utils.py:278] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 8192, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'enable_lora': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


WARNING 06-01 01:02:03 [envs.py:2039] Unknown vLLM environment variable detected: VLLM_DISABLE_DEEP_GEMM


INFO 06-01 01:02:03 [model.py:617] Resolved architecture: Qwen3ForCausalLM


INFO 06-01 01:02:03 [model.py:1752] Using max model len 8192


INFO 06-01 01:02:03 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.


...

(EngineCore pid=305) 

INFO 06-01 01:03:56 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


Phase 3 ready.


(EngineCore pid=305) 

WARNING 06-01 01:08:10 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


In [3]:
# ============ PHASE 3 / CELL J: eval base + every checkpoint on val ============
val_prompts = [prompt_str_for(it) for it in val_items]

candidates = [("base", None)]
for cp in sorted(Path(OUTPUT_DIR).glob("checkpoint-*")):
    candidates.append((cp.name, str(cp)))
if (Path(OUTPUT_DIR) / "final").exists():
    candidates.append(("final", f"{OUTPUT_DIR}/final"))

sp = SamplingParams(n=N_VOTE, temperature=0.7, top_p=0.95, top_k=20,
                    max_tokens=GEN_MAX_TOKENS, repetition_penalty=1.0)

def eval_adapter(name, path, lid):
    req = None if path is None else LoRARequest(name, lid, path)
    outs = llm.generate(val_prompts, sp, lora_request=req)
    n_ok = sum(is_correct(vote([c.text.strip() for c in o.outputs], it), it)
               for it, o in zip(val_items, outs))
    return 100 * n_ok / len(val_items)

results = {}
for lid, (name, path) in enumerate(candidates, start=1):
    acc = eval_adapter(name, path, lid)
    results[name] = (acc, path)
    print(f"{name:>16}: {acc:.2f} %")

best = max(results, key=lambda k: results[k][0])
BEST_ADAPTER = results[best][1]
print(f"\nBest: {best} at {results[best][0]:.2f} %")
print(f"BEST_ADAPTER = {BEST_ADAPTER}")
if best == "base":
    print("WARNING: no checkpoint beat the base model. Will submit base, not adapter.")

Rendering prompts:   0%|          | 0/112 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/336 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   1%|          | 3/336 [00:51<1:35:47, 17.26s/it, est. speed input: 17.50 toks/s, output: 32.08 toks/s]

Processed prompts:   2%|▏         | 6/336 [01:01<49:30,  9.00s/it, est. speed input: 25.73 toks/s, output: 58.01 toks/s]  

Processed prompts:   3%|▎         | 9/336 [01:06<31:02,  5.70s/it, est. speed input: 33.13 toks/s, output: 87.41 toks/s]

...

Processed prompts:  99%|█████████▉| 333/336 [52:15<00:44, 14.67s/it, est. speed input: 37.01 toks/s, output: 426.01 toks/s]

Processed prompts: 100%|██████████| 336/336 [52:17<00:00, 10.47s/it, est. speed input: 37.25 toks/s, output: 433.31 toks/s]

Processed prompts: 100%|██████████| 336/336 [52:17<00:00, 10.47s/it, est. speed input: 37.25 toks/s, output: 433.31 toks/s]

Processed prompts: 100%|██████████| 336/336 [52:17<00:00,  9.34s/it, est. speed input: 37.25 toks/s, output: 433.31 toks/s]

           final: 50.00 %

Best: checkpoint-61 at 50.89 %
BEST_ADAPTER = lora_ckpts/checkpoint-61


In [3]:
# ============ PHASE 3 / CELL K: final inference on private set (resumable) ============
BEST_ADAPTER = globals().get("BEST_ADAPTER", f"{OUTPUT_DIR}/final")
BEST_ADAPTER = "lora_ckpts/checkpoint-61"
print("Using adapter:", BEST_ADAPTER)

priv = [json.loads(l) for l in open(PRIV_PATH)]
SUB_JSONL = "results/private_final.jsonl"
Path(SUB_JSONL).parent.mkdir(parents=True, exist_ok=True)

done = set()
if Path(SUB_JSONL).exists():
    for l in open(SUB_JSONL):
        try: done.add(json.loads(l)["id"])
        except Exception: pass
todo = [d for d in priv if d["id"] not in done]
print(f"{len(done)} done, {len(todo)} to go (of {len(priv)})")

sp  = SamplingParams(n=N_VOTE, temperature=0.7, top_p=0.95, top_k=20,
                     max_tokens=GEN_MAX_TOKENS, repetition_penalty=1.0)
req = None if BEST_ADAPTER is None else LoRARequest("best", 1, BEST_ADAPTER)
CHUNK = 64
with open(SUB_JSONL, "a") as f:
    for s in range(0, len(todo), CHUNK):
        chunk = todo[s:s+CHUNK]
        outs = llm.generate([prompt_str_for(it) for it in chunk], sp, lora_request=req)
        for it, o in zip(chunk, outs):
            chosen = vote([c.text.strip() for c in o.outputs], it)
            f.write(json.dumps({"id": it["id"], "response": chosen}) + "\n")
        f.flush()
        print(f"  {min(s+CHUNK, len(todo))}/{len(todo)}")
print("Private inference complete.")

Using adapter: lora_ckpts/checkpoint-61


320 done, 623 to go (of 943)


Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

WARNING 06-01 01:03:57 [input_processor.py:149] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/192 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:  98%|█████████▊| 138/141 [21:13<00:34, 11.65s/it, est. speed input: 42.07 toks/s, output: 394.54 toks/s]

Processed prompts: 100%|██████████| 141/141 [21:44<00:00, 11.22s/it, est. speed input: 42.29 toks/s, output: 401.22 toks/s]

Processed prompts: 100%|██████████| 141/141 [21:44<00:00, 11.22s/it, est. speed input: 42.29 toks/s, output: 401.22 toks/s]

Processed prompts: 100%|██████████| 141/141 [21:44<00:00,  9.25s/it, est. speed input: 42.29 toks/s, output: 401.22 toks/s]

  623/623
Private inference complete.


In [4]:
# ============ PHASE 3 / CELL L: write submission.csv ============
import csv
priv = [json.loads(l) for l in open(PRIV_PATH)]
order = {d["id"]: k for k, d in enumerate(priv)}
rows = [json.loads(l) for l in open("results/private_final.jsonl")]
rows.sort(key=lambda r: order.get(r["id"], 10**9))

assert len(rows) == len(priv), f"expected {len(priv)} rows, got {len(rows)}"
with open("submission.csv", "w", newline="") as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(["id", "response"])
    for r in rows:
        w.writerow([r["id"], r["response"]])
print(f"Wrote submission.csv with {len(rows)} rows")

Wrote submission.csv with 943 rows
